# Pandas - Parte 2

Nelle lezioni precedenti abbiamo visto le funzionalità di base di Pandas. Oggi vediamo alcune funzionalità più avanzate, fra cui quelle che ci permettono di combinare i contenuti di più DataFrame in uno.

Prima di partire, come al solito, importiamo pandas:

In [ ]:
import pandas as pd

## Chained indexing

Ricominciamo da dove eravamo rimasti, con il nostro DataFrame di film dello studio Ghibli, con la colonna "won oscar" che conteneva un errore:

In [ ]:
ghibli_movies = pd.DataFrame({
    "title" : ["Princess Mononoke", "Grave of the Fireflies", "Porco Rosso", 
               "Tales from Earthsea", "Spirited Away"],
    "director" : ["Hayao Miyazaki", "Isao Takahata", "Hayao Miyazaki", "Gorō Miyazaki", "Hayao Miyazaki"],
    "saddest movie ever?" : [False, True, False, False, False],
    "year": [1997, 1988, 1992, 2006, 2001],
    "won oscar": [False, False, False, False, False]
    },
    index = [1997, 1988, 1992, 2006, 2001]
)

ghibli_movies

Proviamo ad assegnare la colonna "won oscar" con il valore corretto:

In [ ]:
ghibli_movies[ghibli_movies["title"]=="Spirited Away"]["won oscar"] = True

ghibli_movies

Cerchiamo di capire questo errore: abbiamo fatto quello che si chiama *chained indexing*. Infatti se notate abbiamo usato due metodi di indicizzazione per il nostro DataFrame: prima chiediamo le righe dove il titolo è == a "Spirited Away", poi indicizziamo la colonna "won oscar". Questo crea **una copia** del nostro DataFrame, e infatti non abbiamo modificato l'originale!

Per risolvere questo problema, possiamo usare `.loc` quando vogliamo riassegnare un valore del DataFrame

In [ ]:
ghibli_movies.loc[2001, "won oscar"] = True

ghibli_movies

Oppure possiamo usare `.at`

In [ ]:
ghibli_movies["won oscar"] = False
ghibli_movies.at[2001, "won oscar"] = True

ghibli_movies

## Sorting

Una delle operazioni che non abbiamo ancora discusso è quella di **ordinamento** (*sorting*) dei DataFrame. Possiamo infatti riordinare gli assi del nostro DataFrame usando il metodo `.sort_values`.

In [ ]:
ghibli_movies = pd.DataFrame({
    "title" : ["Princess Mononoke", "Grave of the Fireflies", "Porco Rosso", "Tales from Earthsea", "Spirited Away"],
    "director" : ["Hayao Miyazaki", "Isao Takahata", "Hayao Miyazaki", "Gorō Miyazaki", "Hayao Miyazaki"],
    "saddest movie ever?" : [False, True, False, False, False],
    "year": [1997, 1988, 1992, 2006, 2001],
    "won oscar": [False, False, False, False, True]
    }
)
ghibli_movies

In [ ]:
ghibli_movies.sort_values("year")

Attenzione però: la variabile originale `ghibli_movies` non è stata modificata

In [ ]:
ghibli_movies

Per applicare i cambiamenti alla nostra variabile, usiamo il parametro opzionale `inplace`:

In [ ]:
ghibli_movies.sort_values("year", inplace=True)
ghibli_movies

**Oppure** riassegnamo la variabile:

In [ ]:
ghibli_movies = ghibli_movies.sort_values("year")
ghibli_movies

A questo punto però abbiamo "incasinato" gli indici del nostro DataFrame. Per evitare questa situazione, possiamo usare `ignore_index` quando facciamo `sort`:

In [ ]:
ghibli_movies = ghibli_movies.sort_values("year", ignore_index=True)
ghibli_movies

## Una breve nota su `index`

Quando salviamo file usando Pandas, la libreria inserisce anche gli indici come colonna. Vediamo subito il risultato in un file CSV:

In [ ]:
ghibli_movies.to_csv("output/ghibli.csv")

Se non vogliamo includere gli indici, possiamo usare il parametro opzionale `index`:

In [ ]:
ghibli_movies.to_csv("output/ghibli.csv", index=False)

## Quiz

1. Una persona malvagia ha messo in disordine il file che contiene tutte le informazioni sugli Oscar! Aprilo (in `data/academyawards_mess.csv`) usando Pandas e riordina le righe usando `award_year`. Salva il file in formato CSV in `outputs/academyawards_fixed.csv` non inserendo gli indici.

# Liste dentro alle celle

Oltre a strutture puramente bidimensionali, Pandas ci consente anche di creare strutture più complesse (annidate). Ce ne sarebbero molti tipi, ma noi vediamo solo una versione, quella in cui creiamo una lista all'interno di una cella del DataFrame. Per farlo, usiamo il metodo "split" su stringhe, che divide una stringa usando un separatore

In [ ]:
academyawards = pd.read_csv("data/academyawards.csv")

academyawards["directors"] = academyawards["directors"].str.split(",")

Per accedere all'ennesimo elemento di queste liste scriviamo:

In [ ]:
academyawards.loc[0,"directors"][0]

Possiamo assegnare una lista a una cella usando `.at`, `.loc` non funziona!

In [ ]:
# this does not work
academyawards.loc[0,"directors"] = ["Some", "Directors"]

In [ ]:
academyawards.at[0,"directors"] = ["Some", "Directors"]
academyawards.head()

# groupby

Un altro metodo utile di Pandas è groupby. Questo metodo ci consente di "raggruppare" il DataFrame usando i valori di una colonna specifica. Vediamo un esempio:

In [ ]:
animals = pd.DataFrame({'Animal': ['Falcon', 'Falcon',
                              'Parrot', 'Parrot'],
                   'Max Speed': [380., 370., 24., 26.]})
animals.groupby("Animal")

`groupby` ha ora raggruppato le righe tramite i valori della colonna `Animal`. Tuttavia, non possiamo ancora usare questo risultato. Dobbiamo invece usare un metodo per dire a Pandas come aggregare i valori (in questo caso le velocità massime)

In [ ]:
animals.groupby("Animal").mean()

Possiamo anche contare quante volte appare ciascun animale!

In [ ]:
animals.groupby("Animal").count()

Cosa succede se abbiamo anche colonne con valori non numerici?

In [ ]:
meteo_bolo = pd.read_csv("data/meteo_bologna.csv")
meteo_bolo.groupby("Stagione").mean()

Dobbiamo filtrare e usare solo le colonne che ci servono!

In [ ]:
meteo_bolo_stagioni = meteo_bolo[["Stagione", "Precipitazioni (mm)", "Temperatura media"]]
meteo_bolo_stagioni.groupby("Stagione").mean()

## concat

Oltre alla possibilità di aggiungere righe e colonne, Pandas offre una serie di metodi per combinare più DataFrame in uno. Il primo è `concat`, che concatena le righe di un DataFrame ad un altro.

In [ ]:
ghibli_1 = pd.DataFrame({
    "title": ["Princess Mononoke", "Grave of the Fireflies"],
    "director" : ["Hayao Miyazaki", "Isao Takahata"],
    "year": [1997, 1988]
})

ghibli_2 = pd.DataFrame({
    "title": ["Porco Rosso", "Tales from Earthsea"],
    "director": ["Hayao Miyazaki", "Gorō Miyazaki"],
    "year": [1992, 2006]
})

ghibli_movies = pd.concat((ghibli_1, ghibli_2))
ghibli_movies

Se vogliamo risistemare gli indici usiamo `ignore_index`:

In [ ]:
ghibli_movies = pd.concat((ghibli_1, ghibli_2), ignore_index=True)
ghibli_movies

Cosa succede se abbiamo colonne in una tabella ma nell'altra non ci sono?

In [ ]:
ghibli_1["won oscar"]=False
ghibli_movies = pd.concat((ghibli_1, ghibli_2), ignore_index=True)
ghibli_movies

## join


Un'altra posssibilità è quella di voler unire le colonne di due DataFrame. Lo si fa con `join`

In [ ]:
names = pd.DataFrame({"names": ["Andrea", "Claudia", "Piero", "Mafalda"], 
                      "surnames": ["Esposito", "Testa", "Barbieri", "Gallo"]})
ages = pd.DataFrame({"ages": [64, 30, 43, 88]})

names.join(ages)

Cosa succede se la stessa colonna appare più volte?

In [ ]:
names = pd.DataFrame({"names": ["Andrea", "Claudia", "Piero", "Mafalda"],
                      "surnames": ["Esposito", "Testa", "Barbieri", "Gallo"]})
ages = pd.DataFrame({"names": ["Andrea", "Claudia", "Piero", "Mafalda"], 
                     "ages": [64, 30, 43, 88]})
names.join(ages)

Non possiamo farlo! Le colonne avrebbero lo stesso nome! Se vogliamo fare `merge` dobbiamo specificare un suffisso per una o entrambe le colonne

In [ ]:
names = pd.DataFrame({"names": ["Andrea", "Claudia", "Piero", "Mafalda"], 
                      "surnames": ["Esposito", "Testa", "Barbieri", "Gallo"]})
ages = pd.DataFrame({"names": ["Andrea", "Claudia", "Piero", "Mafalda"], 
                     "ages": [64, 30, 43, 88]})
names.join(ages, lsuffix="-left", rsuffix="-right")

## merge

Infine, possiamo "unire" due DataFrame. In questo caso, il sistema controllerà dove le colonne che sono presenti in entrambi i DataFrame hanno lo stesso valore, unendo le que righe corrispondenti:

In [ ]:
names = pd.DataFrame({"names": ["Andrea", "Claudia", "Piero", "Mafalda"], 
                      "surnames": ["Esposito", "Testa", "Barbieri", "Gallo"]})
ages = pd.DataFrame({"names": ["Andrea", "Claudia", "Piero", "Mafalda"], 
                     "ages": [64, 30, 43, 88]})
pd.merge(names, ages)

In [ ]:
# what happens when there are rows on one side and not the other?
names = pd.DataFrame({"names": ["Andrea", "Claudia", "Piero", "Mafalda", "Giovanni"], 
                      "surnames": ["Esposito", "Testa", "Barbieri", "Gallo", "Serra"]})
ages = pd.DataFrame({"names": ["Andrea", "Claudia", "Piero", "Mafalda", "Antonio"], 
                     "ages": [64, 30, 43, 88, 24]})
pd.merge(names, ages)

In [ ]:
# or i could do an outer merge
names = pd.DataFrame({"names": ["Andrea", "Claudia", "Piero", "Mafalda", "Giovanni"], 
                      "surnames": ["Esposito", "Testa", "Barbieri", "Gallo", "Serra"]})
ages = pd.DataFrame({"names": ["Andrea", "Claudia", "Piero", "Mafalda", "Antonio"], 
                     "ages": [64, 30, 43, 88, 24]})
pd.merge(names, ages, how="outer")

## Quiz

1. Scarica i dati di temperatura e precipitazioni (disaggregati) da [qui](https://opendata.comune.bologna.it/explore/?sort=modified&refine.theme=Ambiente). Unisci i due file CSV usando Pandas.